In [ ]:
import os
import torch
import torchvision
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import numpy as np
from tqdm import tqdm
import random
from torchvision.transforms import functional as F
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

DATA_DIR = './data'
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_EPOCHS = 5

class VOCSegmentationCustom(torch.utils.data.Dataset):
    def __init__(self, root, year, image_set, download=True):
        self.base_dataset = datasets.VOCSegmentation(
            root=root, year=year, image_set=image_set,
            download=download, transforms=None
        )
        self.is_train = (image_set == 'train')

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        img = Image.open(self.base_dataset.images[idx]).convert('RGB')
        mask = Image.open(self.base_dataset.masks[idx])

        img = F.resize(img, IMAGE_SIZE, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.resize(mask, IMAGE_SIZE, interpolation=F.InterpolationMode.NEAREST)

        if self.is_train and random.random() > 0.5:
            img = F.hflip(img)
            mask = F.hflip(mask)

        img = F.to_tensor(img)
        img = F.normalize(img, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask = torch.as_tensor(np.array(mask), dtype=torch.int64)

        return img, mask

train_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='train', download=True)
val_dataset = VOCSegmentationCustom(root=DATA_DIR, year='2012', image_set='val', download=True)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

Device: cuda


100%|██████████| 2.00G/2.00G [01:12<00:00, 27.5MB/s]


Train: 1464, Val: 1449


In [2]:
from torchvision.models.segmentation import deeplabv3_resnet50, DeepLabV3_ResNet50_Weights

model = deeplabv3_resnet50(weights=DeepLabV3_ResNet50_Weights.DEFAULT)
model.to(device)

criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.0005)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

print(f"Training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}"):
        images, masks = images.to(device), masks.to(device)

        optimizer.zero_grad()
        outputs = model(images)['out']
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    scheduler.step()
    print(f"Epoch {epoch+1} completed. Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), 'model.pth')
print("Model saved!")

Downloading: "https://download.pytorch.org/models/deeplabv3_resnet50_coco-cd0a2569.pth" to /root/.cache/torch/hub/checkpoints/deeplabv3_resnet50_coco-cd0a2569.pth


100%|██████████| 161M/161M [00:01<00:00, 94.1MB/s]


Training for 5 epochs...


Epoch 1/5: 100%|██████████| 183/183 [02:11<00:00,  1.39it/s]


Epoch 1 completed. Loss: 0.7648


...

Epoch 3 completed. Loss: 0.3819


Epoch 4/5: 100%|██████████| 183/183 [02:28<00:00,  1.23it/s]


Epoch 4 completed. Loss: 0.3008


Epoch 5/5: 100%|██████████| 183/183 [02:29<00:00,  1.23it/s]


Epoch 5 completed. Loss: 0.3034
Model saved!


In [3]:
def calculate_iou(pred, target):
    unique_classes = torch.unique(target)
    ious = []
    for cls in unique_classes:
        if cls == 255:
            continue
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union > 0:
            ious.append(intersection / union)
    return ious

random.seed(42)
all_indices = list(range(len(val_dataset)))
selected_indices = random.sample(all_indices, 100)

results = []
all_mean_ious = []

model.eval()
with torch.no_grad():
    for idx in tqdm(selected_indices, desc="Evaluating"):
        img = Image.open(val_dataset.base_dataset.images[idx]).convert('RGB')
        mask = Image.open(val_dataset.base_dataset.masks[idx])
        img_name = os.path.basename(val_dataset.base_dataset.masks[idx])

        img = F.resize(img, IMAGE_SIZE, interpolation=F.InterpolationMode.BILINEAR)
        mask = F.resize(mask, IMAGE_SIZE, interpolation=F.InterpolationMode.NEAREST)

        img_tensor = F.to_tensor(img)
        img_tensor = F.normalize(img_tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        mask_tensor = torch.as_tensor(np.array(mask), dtype=torch.int64)

        output = model(img_tensor.unsqueeze(0).to(device))['out']
        pred = output.argmax(dim=1).squeeze(0).cpu()

        ious = calculate_iou(pred, mask_tensor)
        mean_iou = sum(ious) / len(ious) if ious else 0.0

        all_mean_ious.append(mean_iou)
        results.append({
            'name': img_name,
            'mean_iou': mean_iou,
            'num_objects': len(ious),
            'ious': ious
        })

global_mean_iou = sum(all_mean_ious) / len(all_mean_ious)

with open("results.txt", "w") as f:
    f.write(f"{global_mean_iou:.6f}\n")
    for res in results:
        ious_str = " ".join([f"{iou:.6f}" for iou in res['ious']])
        f.write(f"{res['name']} {res['mean_iou']:.6f} {res['num_objects']} {ious_str}\n")

print(f"Mean IoU: {global_mean_iou:.6f}")
print("File results.txt created!")

Evaluating: 100%|██████████| 100/100 [00:04<00:00, 22.06it/s]

Mean IoU: 0.589462
File results.txt created!
